# 1. Récupération et préparation des données

**Étape 1 sur 5.** Ce notebook télécharge (une seule fois : les fichiers sont ensuite gardés dans `data/`) et prépare les données de toute l'analyse. Il est exécuté automatiquement par les autres notebooks.

Variables produites : `installations_indust`, `classement_dpt`, `DEPARTEMENTS`, `stations`, `chroniques`, `chroniques_niveau_mensuel`, `requete_hubeau`.

**Organisation des notebooks.**

| Notebook | Contenu | Exécute d'abord |
|---|---|---|
| `eau_01_donnees.ipynb` | Récupération et préparation des données | — |
| `eau_02_usines_stations.ipynb` | Usines et stations : premier regard | 01 |
| `eau_03_geographie.ipynb` | Altitude, roche, profondeur | 01 |
| `eau_04_climat.ipynb` | Retrait de l'effet de la pluie | 03 |
| `eau_05_industrie.ipynb` | Effet de l'industrie | 01 |
| `rapport.ipynb` | Figures et interprétations uniquement | 02, 04, 05 |

Chaque notebook exécute ceux dont il a besoin avec `%run`, sans afficher leurs sorties et une seule fois par noyau : pour tout recalculer après une modification, redémarrer le noyau.

In [ ]:
import os
import ast
import time

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import requests as rq
import contextily as ctx
from scipy.stats import spearmanr
from matplotlib.colors import TwoSlopeNorm

#AFFICHER vaut True quand on ouvre ce notebook seul, et False quand un autre notebook l'exécute (étape suivante ou rapport) :
#dans ce cas, on fait tous les calculs mais on ne trace aucune figure
AFFICHER = globals().get("AFFICHER", True)

## 1.1 Installations industrielles (ICPE)
On utilise l'API de Géorisques pour récupérer les informations sur les usines.

In [ ]:
#On crée le dossier data qui va stocker toutes nos données :
os.makedirs("../../data", exist_ok=True)

#On récupère la liste des installations industrielles à l'échelle nationale sur le site de l'ICPE
url_icpe = "https://mapsref.brgm.fr/wxs/georisques/georisques_dl?&service=wfs&version=2.0.0&request=getfeature&typename=InstallationsClassees&outputformat=CSVTEXT"

lien_stockage_physique_icpe = "../../data/icpe.csv"

#Si le fichier a déjà été téléchargé, on réutilise la copie locale (le téléchargement est long)
if os.path.exists(lien_stockage_physique_icpe):
    print("Fichier ICPE déjà téléchargé, on réutilise la copie locale.")
else:
    fichier_icpe = rq.get(url_icpe)

    if fichier_icpe.status_code == 200:
        with open(lien_stockage_physique_icpe, "wb") as f:
            f.write(fichier_icpe.content)
        print("Fichier téléchargé.")
    else:
        print("Erreur avec le téléchargement du fichier ICPE.")

In [ ]:
# On récupère donc la liste des installations industrielles
installations_indust = pd.read_csv(lien_stockage_physique_icpe, sep=";", encoding='utf-8')
print("On a ", installations_indust.shape[0], "installations industrielles.", end="")
installations_indust = installations_indust.loc[installations_indust["lib_regime"] != "Non ICPE"]
print(" Dont ", installations_indust.shape[0], "installations industrielles ICPE.")

#On veut garder uniquement les installations industrielles pures (pas l'agriculture et l'élevage par ex)
installations_indust = (installations_indust.loc[
    (installations_indust["industrie"] == 1)
    & (installations_indust["bovins"] == 0)
    & (installations_indust["porcs"] == 0)
    & (installations_indust["volailles"] == 0)
    & (installations_indust["eolienne"] == 0)
    ])
print("On ne garde que les ", installations_indust.shape[0], "installations industrielles pures.")

#On classe les départements qui ont le plus d'installations industrielles pour limiter la recherche (car les données sur Hub'eau sont trop volumineuses pour traiter la France entière)
classement_dpt = (installations_indust.groupby("num_dep")
    .size()
    .sort_values(ascending=False)
    )

In [ ]:
def fig_classement_departements():
    """Nombre d'installations industrielles par département, du plus industriel au moins industriel."""
    plt.figure()
    plt.bar(range(classement_dpt.shape[0]), classement_dpt.values)
    plt.title("Nombre d'installations industrielles dans chaque département")
    plt.xlabel("Rang du département (0 : le département avec le plus d'installations)")
    plt.ylabel("Nombre d'installations industrielles")
    plt.show()

if AFFICHER:
    fig_classement_departements()

On prend par défaut le département qui contient le plus d'installations industrielles pour notre analyse.  
En effet, l'API Hub'eau nous donne un très grand volume de données et nous devons nous limiter dans notre recherche à quelques départements.  
Avec le graphique ci-dessus, on voit que le département 59 est de loin le plus grand centre industriel français.  
Nous nous concentrerons donc sur le département 59 pour la suite de l'analyse.  
La liste `DEPARTEMENTS` ci-dessous permet d'en étudier plusieurs à la fois : toute la suite de l'analyse s'applique à tous les départements de la liste,
et chaque département n'est téléchargé qu'une fois (fichiers `data/..._<département>.csv`).

In [ ]:
#Départements étudiés : par défaut celui qui a le plus d'ICPE (le 59)
#Pour en étudier plusieurs, il suffit de compléter la liste, par exemple : DEPARTEMENTS = ["59", "88", "68"]
DEPARTEMENTS = [str(classement_dpt.index[0])]
print("Départements analysés : ", DEPARTEMENTS)

Mise en forme des données géographiques avec Geopandas

In [ ]:
installations_indust = installations_indust.loc[installations_indust["num_dep"].isin(DEPARTEMENTS)]

#On vérifie qu'il n'y a pas d'installations industrielles sans coordonnées géographiques dans la base de données ICPE des installations industrielles des départements étudiés
nb_position_less = installations_indust.loc[installations_indust["x"].isnull() | installations_indust["y"].isnull() | installations_indust["code_epsg"].isnull()].shape[0]
print(nb_position_less, "installations industrielles n'ont pas de coordonnées géographiques.")
installations_indust = installations_indust.loc[installations_indust["x"].notnull() & installations_indust["y"].notnull() & installations_indust["code_epsg"].notnull()]

#On utilise les coordonnées des installations pour les localiser avec Geopandas
installations_indust["geometry"] = None #Colonne qui va prendre la valeur position Geopandas
codes_epsg_type_coord = installations_indust["code_epsg"].unique()   #On veut que ça marche pour les différents types de référentiels (x,y)
for epsg in codes_epsg_type_coord:
    filtre = installations_indust["code_epsg"] == epsg
    points = gpd.points_from_xy(
        installations_indust.loc[filtre, "x"], 
        installations_indust.loc[filtre, "y"]
    )
    serie_geometrique = gpd.GeoSeries(points, crs=f"EPSG:{int(epsg)}")
    serie_geometrique = serie_geometrique.to_crs("EPSG:2154") #On ramène tout sous le même format
    installations_indust.loc[filtre, "geometry"] = serie_geometrique.values

#On définit un GeoDataFrame pour les études géométriques :
installations_indust = gpd.GeoDataFrame(installations_indust, geometry="geometry", crs="EPSG:2154")

## 1.2 Nappes phréatiques des départements étudiés

L'API Hub'eau ne permet pas d'envoyer toutes les stations d'un coup.  
Encore moins toutes les chroniques d'un coup.  
Donc on ne sélectionne que les départements étudiés (par défaut le 59, où l'on a vu qu'il y avait une grande concentration d'industrie).

In [ ]:
morceaux = []
for dep in DEPARTEMENTS:
    lien_stations = f"../../data/stations_nappes_{dep}.csv"   #Un fichier en cache par département
    if not os.path.exists(lien_stations):
        #On récupère les stations de mesure des nappes phréatiques sur le site Hubeau
        url_stations = "https://hubeau.eaufrance.fr/api/v1/niveaux_nappes/stations"
        params = {"code_departement": dep, "size": 5000, "format": "json"}
        fichier_stations = rq.get(url_stations, params=params)

        if fichier_stations.status_code == 200:
            pd.DataFrame(fichier_stations.json()["data"]).to_csv(lien_stations, encoding='utf-8')
            print("Stations du département", dep, "téléchargées.")
        else:
            print(fichier_stations.status_code)
            print("Erreur avec le téléchargement du fichier stations Hubeau.")
    morceaux.append(pd.read_csv(lien_stations, index_col=0).assign(departement=dep))   #On garde le département étudié de chaque station
stations = pd.concat(morceaux, ignore_index=True)

#Dans le CSV, les listes (masses d'eau, entités BD LISA) sont stockées en texte : on les reconvertit en listes
for colonne in ["codes_bdlisa", "urns_bdlisa", "codes_masse_eau_edl", "noms_masse_eau_edl", "urns_masse_eau_edl"]:
    stations[colonne] = stations[colonne].apply(lambda v: ast.literal_eval(v) if isinstance(v, str) else v)
print("On a ", stations.shape[0], "stations de mesure des nappes phréatiques dans les départements ", DEPARTEMENTS, ".")

Mise en forme des données géographiques avec Geopandas

In [ ]:
#On vérifie qu'il n'y a pas de stations sans coordonnées géographiques
nb_position_less = stations.loc[stations["x"].isnull() | stations["y"].isnull()].shape[0]
print(nb_position_less, "stations n'ont pas de coordonnées géographiques.")
stations = stations.loc[stations["x"].notnull() & stations["y"].notnull()]

#On crée le modèle Geopandas
stations = gpd.GeoDataFrame(
    stations,
    geometry=gpd.points_from_xy(stations['x'], stations['y']),
    crs="EPSG:4326"
).to_crs(epsg=2154) #On ramène tout sous le même format que les installations industrielles

Pour chaque station des départements étudiés, on récupère les séries temporelles de niveau des nappes phréatiques entre janvier 2015 et janvier 2026.

In [ ]:
#On ne peut pas récupérer les mesures des nappes phréatiques pour toutes les stations en même temps.
#Quand on essaye de récupérer toutes les données de toutes les nappes, on a une erreur 414 : la requête URL est trop longue avec tous les code_bss en paramètres.
#Et Hub'eau refuse d'aller au-delà de 20 000 lignes par requête (page x taille de page <= 20 000) : avec plus, il répond une erreur 400.
#Donc on demande les stations par petits paquets, et on recoupe un paquet trop gros en deux (puis la période en deux pour une seule station).
url_chroniques_api = "https://hubeau.eaufrance.fr/api/v1/niveaux_nappes/chroniques"

def requete_hubeau(params, url=url_chroniques_api):
    """Une requête Hub'eau, réessayée en cas de coupure ou de serveur saturé (mais pas en cas de requête refusée)."""
    for essai in range(10):
        try:
            reponse = rq.get(url, params=params, timeout=180)
        except Exception as e:
            print("Erreur requête :", e, "- pas grave, on va réessayer.")
            time.sleep(2)
            continue
        if reponse.status_code in (200, 206) and reponse.headers.get("content-type", "").startswith("application/json"):
            return reponse.json()
        if 400 <= reponse.status_code < 500 and reponse.status_code != 429:   #Requête refusée : réessayer ne servirait à rien
            raise RuntimeError(f"Hub'eau a refusé la requête ({reponse.status_code}) : {reponse.text[:200]}")
        time.sleep(2 * (essai + 1))   #Serveur saturé : on attend un peu plus à chaque fois
    raise RuntimeError("Hub'eau ne répond pas.")

def telecharger_paquet(paquet, debut="2015-01-01", fin="2026-01-01"):
    data = requete_hubeau({"code_bss": ",".join(paquet), "size": 20000, "format": "json", "date_debut_mesure": debut, "date_fin_mesure": fin})
    if data["count"] <= 20000:   #Tout tient dans une seule page
        return data["data"]
    if len(paquet) > 1:   #Trop de lignes : on coupe le paquet de stations en deux
        return telecharger_paquet(paquet[:len(paquet) // 2], debut, fin) + telecharger_paquet(paquet[len(paquet) // 2:], debut, fin)
    milieu = str(pd.Timestamp(debut) + (pd.Timestamp(fin) - pd.Timestamp(debut)) / 2)[:10]   #Une seule station : on coupe la période en deux
    return telecharger_paquet(paquet, debut, milieu) + telecharger_paquet(paquet, milieu, fin)

def telecharger_chroniques(liste_codes_bss_stations):
    chroniques = []
    for i in range(0, len(liste_codes_bss_stations), 5):   #Paquets de 5 stations : environ 4 000 mesures par station sur 11 ans
        chroniques.extend(telecharger_paquet(liste_codes_bss_stations[i:i+5]))
        print("Stations téléchargées :", min(i + 5, len(liste_codes_bss_stations)), "sur", len(liste_codes_bss_stations))
    #Les deux moitiés d'une période coupée en deux partagent leur jour de coupure : on enlève ces doublons
    return pd.DataFrame(chroniques).drop_duplicates(subset=["code_bss", "date_mesure", "timestamp_mesure"])


morceaux = []
for dep in DEPARTEMENTS:
    lien_chroniques = f"../../data/chroniques_nappes_{dep}.csv"   #Un fichier en cache par département
    if not os.path.exists(lien_chroniques):
        #On ne télécharge que les stations qui peuvent être mesurées au moins 80 % des mois de 2015-2025 (critère des parties 3 et 4) :
        #début des mesures au plus tard en mars 2017 et fin au plus tôt en octobre 2023. Les autres seraient écartées de toute façon.
        utiles = stations.loc[(stations["departement"] == dep) & (stations["date_debut_mesure"] <= "2017-03-31") & (stations["date_fin_mesure"] >= "2023-10-01")]
        telecharger_chroniques(utiles["code_bss"].tolist()).to_csv(lien_chroniques, encoding='utf-8')
        print("Chroniques du département", dep, "téléchargées.")
    morceaux.append(pd.read_csv(lien_chroniques, index_col=0))
chroniques = pd.concat(morceaux, ignore_index=True)
print("Chroniques chargées :", chroniques.shape[0], "mesures.")

On nettoie les données des séries temporelles des nappes phréatiques.

In [ ]:
chroniques['date_mesure'] = pd.to_datetime(chroniques['date_mesure'])
print("Nombre de chroniques sans niveau d'eau : ", chroniques.loc[chroniques["niveau_nappe_eau"].isnull()].shape[0])
#On supprime les chroniques pour lesquelles le niveau d'eau est NULL
chroniques = chroniques.loc[chroniques["niveau_nappe_eau"].notnull()]

## 1.3 Test des données : exemple de série temporelle

Un exemple de série temporelle pour une station (nappe phréatique).

In [ ]:
def fig_exemple_serie(station_cible='00285X0478/PZ10'):
    """Série temporelle brute du niveau d'une station."""
    df_station = chroniques[chroniques['code_bss'] == station_cible]
    df_station = df_station.sort_values(by='date_mesure')

    plt.figure(figsize=(10, 5))
    plt.plot(
        df_station['date_mesure'], 
        df_station['niveau_nappe_eau'], 
        marker='o',         # Ajoute un point sur chaque mesure réelle
        linestyle='-',      # Relie les points par une ligne
        color='#1f77b4', 
        linewidth=2
    )

    plt.title(f"Évolution du niveau de la nappe - Station {station_cible}", fontsize=14)
    plt.xlabel("Date de mesure", fontsize=12)
    plt.ylabel("Niveau de la nappe d'eau (m)", fontsize=12)
    plt.xticks(rotation=45) # Penche les dates pour éviter qu'elles ne se superposent
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

if AFFICHER:
    fig_exemple_serie()

## 1.4 Nettoyage et standardisation des données

Les séries temporelles de Hub'eau donnent des mesures des nappes phréatiques à des dates différentes entre les stations.  
Pour les calculs futurs, on va standardiser ça en faisant une moyenne par station et par mois.

In [ ]:
chroniques['date_mois_annee'] = chroniques['date_mesure'].dt.to_period('M')

chroniques_niveau_mensuel = (chroniques.groupby(["code_bss", "date_mois_annee"])
              .agg({
                  "niveau_nappe_eau":"mean",
                  "profondeur_nappe":"mean"
              })
              .reset_index()
    )
chroniques_niveau_mensuel["date_mesure"] = chroniques_niveau_mensuel["date_mois_annee"].dt.to_timestamp()
chroniques_niveau_mensuel['date_mesure'] = pd.to_datetime(chroniques_niveau_mensuel['date_mesure'])
print("Les données ont été normalisées en moyenne par mois pour chaque station.")

In [ ]:
#Marqueur : cette étape est faite, les notebooks qui en dépendent ne la relanceront pas dans ce noyau
ETAPE_01 = True